Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, date
from scipy import stats
from scipy.stats import linregress

File paths + global variables

In [2]:
DATA_DIR   = Path('.')
EXCEL_FILE = DATA_DIR / 'data.xlsx'
TEMPLATE   = DATA_DIR / 'template_forecast_v00.csv'
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

PORTFOLIOS          = ['A', 'B', 'C', 'D']
FORECAST_MONTH      = 8
FORECAST_YEAR       = 2025
TRAINING_MONTHS     = ['April', 'May', 'June']
TRAINING_MONTH_NUMS = [4, 5, 6]

MONTH_NAMES  = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
                7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
DOW_NAMES    = {0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'}
MONTH_TO_NUM = {v:k for k,v in MONTH_NAMES.items()}
ALL_SLOTS    = list(range(0, 24*60, 30)) 
SLOT_LABELS  = [f"{m//60}:{m%60:02d}" for m in ALL_SLOTS]


TRIM_FRAC = 0.10 

# global variables for overnight cutoff times for business day start and end
OVERNIGHT_CUTOFF_AM = 7 * 60    
OVERNIGHT_CUTOFF_PM = 22 * 60  

Functions to load all the data and parse it 

In [3]:
def parse_date(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)): return None
    try: return datetime.strptime(str(raw).strip().split()[0], '%m/%d/%y').date()
    except: return None

def to_mins(val):
    if hasattr(val, 'hour'): return int(val.hour * 60 + val.minute)
    if isinstance(val, float): return None if np.isnan(val) else int(round(val * 24 * 60))
    if isinstance(val, str):
        try: p = val.strip().split(':'); return int(p[0]) * 60 + int(p[1])
        except: return None
    return None

def load_daily(p):
    df = pd.read_excel(EXCEL_FILE, sheet_name=f'{p} - Daily', engine='openpyxl')
    df.columns = df.columns.str.strip()
    df['date'] = df[df.columns[0]].apply(parse_date)
    df = df.dropna(subset=['date']).copy()
    df['year']         = df['date'].apply(lambda d: d.year)
    df['month_num']    = df['date'].apply(lambda d: d.month)
    df['day_of_month'] = df['date'].apply(lambda d: d.day)
    df['dow']          = df['date'].apply(lambda d: d.weekday())
    df['month_name']   = df['month_num'].map(MONTH_NAMES)
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    return df.sort_values('date').reset_index(drop=True)

def load_interval(p):
    df = pd.read_excel(EXCEL_FILE, sheet_name=f'{p} - Interval', engine='openpyxl')
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'Month':'month_name','Day':'day_of_month','Interval':'interval_raw'})
    df['mins'] = df['interval_raw'].apply(to_mins)
    df['month_num'] = df['month_name'].map(MONTH_TO_NUM)
    df = df[df['mins'].notna()].copy()
    df['mins'] = df['mins'].astype(int)
    return df.sort_values(['month_num','day_of_month','mins']).reset_index(drop=True)

daily    = {p: load_daily(p)    for p in PORTFOLIOS}
interval = {p: load_interval(p) for p in PORTFOLIOS}

#Slots with max calls
PEAK_SLOTS = set(range(540, 1050, 30))
RAMP_SLOTS = set(range(420, 540, 30))

Detecting a corrupt data value 

corrupt day is when partial/no values are found - forming the basis for exclusion of data from the model to get better forecasting values

In [4]:
def get_clean_interval(df_int, df_daily_p):
    
    daily_lk = df_daily_p[
        (df_daily_p['year'] == 2025) &
        (df_daily_p['month_name'].isin(TRAINING_MONTHS))
    ].set_index(['month_name','day_of_month'])['Call Volume'].to_dict()

    exclude = set()
    for (month, day), grp in df_int.groupby(['month_name','day_of_month']):
        valid = set(grp['mins'].tolist())
        if (PEAK_SLOTS - valid) or len(RAMP_SLOTS - valid) > 2:
            exclude.add((month, int(day))); continue
        dcv = daily_lk.get((month, int(day)), None)
        if dcv and dcv > 0 and grp['Call Volume'].sum() < 0.90 * dcv:
            exclude.add((month, int(day)))

    df_clean = df_int[
        ~df_int.apply(lambda r: (r['month_name'], int(r['day_of_month'])) in exclude, axis=1)
    ].copy().reset_index(drop=True)
    n_clean = df_clean.groupby(['month_name','day_of_month']).ngroups
    print(f"  Excluded {len(exclude)} corrupt days: {n_clean} clean training days")
    return df_clean

clean_int = {}
for p in PORTFOLIOS:
    clean_int[p] = get_clean_interval(interval[p], daily[p])


  Excluded 7 corrupt days: 84 clean training days
  Excluded 5 corrupt days: 86 clean training days
  Excluded 7 corrupt days: 84 clean training days
  Excluded 6 corrupt days: 85 clean training days


In [5]:
#join day of week with interval data
def join_dow(df_int_clean, df_daily_p):
    daily_2025 = df_daily_p[
        (df_daily_p['year'] == 2025) &
        (df_daily_p['month_name'].isin(TRAINING_MONTHS))
    ][['month_name','day_of_month','dow','Call Volume','CCT','Abandon Rate']].copy()

    df = df_int_clean.merge(daily_2025, on=['month_name','day_of_month'], how='inner')
    df = df.rename(columns={
        'Call Volume_x': 'cv_slot', 'Call Volume_y': 'cv_daily',
        'CCT_x': 'cct_slot', 'CCT_y': 'cct_daily',
        'Abandoned Rate': 'abd_slot', 'Abandon Rate': 'abd_daily',
    })
    df['cv_frac'] = (df['cv_slot'] / df['cv_daily'].clip(lower=1)).clip(0, 0.5)
    return df

merged = {p: join_dow(clean_int[p], daily[p]) for p in PORTFOLIOS}


To understand the values and patterns for CV, CCT and ABD by building shape profiles

In [6]:
def trimmed_mean(x, trim=TRIM_FRAC):
    x = np.array(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0: return np.nan
    if len(x) <= 3: return np.mean(x)  
    n_trim = max(1, int(np.floor(len(x) * trim)))
    x_sorted = np.sort(x)
    return x_sorted[n_trim:-n_trim].mean()

#Shape for CV
def build_cv_shape(df_merged):
    shapes = {}
    fallback = df_merged.groupby('mins')['cv_frac'].apply(trimmed_mean)
    fallback = fallback / fallback.sum()

    for dow in range(7):
        dow_df = df_merged[df_merged['dow'] == dow]
        slot_means = dow_df.groupby('mins')['cv_frac'].apply(trimmed_mean)
        slot_means = slot_means.reindex(ALL_SLOTS).fillna(fallback.reindex(ALL_SLOTS)).fillna(1/48)
        total = slot_means.sum()
        shapes[dow] = (slot_means / total if total > 0 else slot_means).to_dict()

    return shapes

#Shape for CCT
def build_cct_shape(df_merged):
    cct_valid = df_merged[df_merged['cct_slot'].between(60, 1200)].copy() 
    global_mean = cct_valid['cct_slot'].mean()                              

    slot_avg_vol = cct_valid.groupby('mins')['cv_slot'].mean()

    shapes = {}
    for dow in range(7):
        dow_df   = cct_valid[cct_valid['dow'] == dow]
        dow_mean = dow_df['cct_slot'].mean() if len(dow_df) > 0 else global_mean  

        slot_vals = {}
        for mn in range(0, 1440, 30):
            slot_df  = dow_df[dow_df['mins'] == mn]
            avg_vol  = slot_avg_vol.get(mn, 0)

            if len(slot_df) >= 5 and avg_vol >= 10:
                slot_vals[mn] = trimmed_mean(slot_df['cct_slot'].values)   
            elif len(slot_df) >= 3:
                slot_vals[mn] = slot_df['cct_slot'].mean()                
            else:
                slot_vals[mn] = dow_mean
        shapes[dow] = slot_vals
    return shapes

#Shape for ABD
def build_abd_shape(df_merged):
    abd_col = 'abd_slot'        
    OVERNIGHT_AM = 7 * 60
    OVERNIGHT_PM = 22 * 60
    SHRINKAGE    = 0.5

    abd_valid = df_merged[
        df_merged[abd_col].between(0, 1) & (df_merged['cv_slot'] >= 5)
    ].copy()

    daytime = abd_valid[
        abd_valid['mins'].between(OVERNIGHT_AM, OVERNIGHT_PM - 1)
    ][abd_col]
    global_mean = daytime.mean() if len(daytime) > 0 else 0.02

    shapes = {}
    for dow in range(7):
        dow_df = abd_valid[abd_valid['dow'] == dow]
        result = {}
        for mn in range(0, 1440, 30):
            if mn < OVERNIGHT_AM or mn >= OVERNIGHT_PM:
                result[mn] = 0.0
                continue
            slot_df = dow_df[dow_df['mins'] == mn]
            if len(slot_df) >= 3:
                slot_mean  = slot_df[abd_col].mean()
                result[mn] = SHRINKAGE * slot_mean + (1 - SHRINKAGE) * global_mean
            else:
                result[mn] = global_mean
        shapes[dow] = result
    return shapes

# Adding all shapes lists
cv_shapes  = {}
cct_shapes = {}
abd_shapes = {}

for p in PORTFOLIOS:
    cv_shapes[p]  = build_cv_shape(merged[p])
    cct_shapes[p] = build_cct_shape(merged[p])
    abd_shapes[p] = build_abd_shape(merged[p])


Creating daily anchors for Aug 2025 data with the populated values

In [7]:
def get_august_anchors(df_daily_p):
    aug25 = df_daily_p[
        (df_daily_p['year'] == 2025) & (df_daily_p['month_num'] == FORECAST_MONTH)
    ][['date','day_of_month','dow','week_of_month',
       'Call Volume','CCT','Service Level','Abandon Rate']].copy().reset_index(drop=True)

    aug24     = df_daily_p[(df_daily_p['year']==2024)&(df_daily_p['month_num']==8)]
    aug24_dow = aug24.groupby('dow')[['Call Volume','CCT','Abandon Rate']].mean()
    known25   = aug25.dropna(subset=['Call Volume'])
    aug25_dow = known25.groupby('dow')[['Call Volume','CCT','Abandon Rate']].mean()

    # calculating the year-on-year ratio for each day of the week (dow)
    yoy_cv = {}
    for dow in aug24_dow.index:
        if dow in aug25_dow.index and aug24_dow.loc[dow,'Call Volume'] > 0:
            yoy_cv[dow] = np.clip(aug25_dow.loc[dow,'Call Volume'] / aug24_dow.loc[dow,'Call Volume'], 0.5, 2.0)
        else:
            yoy_cv[dow] = 1.0

    #removing missing values in CV
    for idx in aug25[aug25['Call Volume'].isna()].index:
        dow = aug25.loc[idx, 'dow']
        if dow in aug25_dow.index:
            aug25.loc[idx, 'Call Volume'] = aug25_dow.loc[dow, 'Call Volume']
            aug25.loc[idx, 'CCT']         = aug25_dow.loc[dow, 'CCT'] if pd.notna(aug25_dow.loc[dow, 'CCT']) else aug24_dow.get(dow, {}).get('CCT', 320)
        elif dow in aug24_dow.index:
            aug25.loc[idx, 'Call Volume'] = aug24_dow.loc[dow, 'Call Volume'] * yoy_cv.get(dow, 1.0)
            aug25.loc[idx, 'CCT']         = aug24_dow.loc[dow, 'CCT']

    # removing missing values in CCT
    for idx in aug25[aug25['CCT'].isna()].index:
        dow = aug25.loc[idx, 'dow']
        aug25.loc[idx, 'CCT'] = aug25_dow.loc[dow, 'CCT'] if dow in aug25_dow.index else aug25['CCT'].median()

    # removing missing values in ABD
    for idx in aug25[aug25['Abandon Rate'].isna()].index:
        dow = aug25.loc[idx, 'dow']
        aug25.loc[idx, 'Abandon Rate'] = aug25_dow.loc[dow, 'Abandon Rate'] if dow in aug25_dow.index else aug25['Abandon Rate'].median()

    return aug25
anchors = {p: get_august_anchors(daily[p]) for p in PORTFOLIOS}
for p in PORTFOLIOS:
    null_cv = anchors[p]['Call Volume'].isna().sum()
    null_cct= anchors[p]['CCT'].isna().sum()
    print(f"  Portfolio {p}: {len(anchors[p])} days; null CV={null_cv}; null CCT={null_cct}")


  Portfolio A: 31 days; null CV=0; null CCT=0
  Portfolio B: 31 days; null CV=0; null CCT=0
  Portfolio C: 31 days; null CV=0; null CCT=0
  Portfolio D: 31 days; null CV=0; null CCT=0


Using the Newsvendors formula to uplift values 

In [8]:
def compute_optimal_uplift(df_daily_p, portfolio):
    train = df_daily_p[
        df_daily_p['month_num'].isin(TRAINING_MONTH_NUMS)
    ].dropna(subset=['Call Volume'])

    residuals = []
    for dow in range(7):
        sub = train[train['dow'] == dow]['Call Volume'].values
        if len(sub) < 2: continue
        for i in range(len(sub)):
            others = np.delete(sub, i)
            pred   = others.mean()
            residuals.append(sub[i] - pred)

    if not residuals:
        return 0.03

    residuals = np.array(residuals)
    mean_cv   = train['Call Volume'].mean()
    q67       = np.quantile(residuals, 2/3) 
    uplift    = float(np.clip(q67 / mean_cv if mean_cv > 0 else 0.03, 0.0, 0.08))

    print(f"  Portfolio {portfolio}: n_residuals={len(residuals)};"
          f"q67={q67:.0f}; mean_CV={mean_cv:.0f}; uplift={uplift:.3f} ({uplift*100:.1f}%)")
    return uplift


optimal_uplift = {}
for p in PORTFOLIOS:
    optimal_uplift[p] = compute_optimal_uplift(daily[p], p)


  Portfolio A: n_residuals=170;q67=123; mean_CV=4118; uplift=0.030 (3.0%)
  Portfolio B: n_residuals=174;q67=522; mean_CV=8984; uplift=0.058 (5.8%)
  Portfolio C: n_residuals=175;q67=881; mean_CV=19802; uplift=0.044 (4.4%)
  Portfolio D: n_residuals=182;q67=621; mean_CV=10420; uplift=0.060 (6.0%)


#### 8. Indentifying Intramonth Trend using OLS regression

In [9]:
def compute_intramonth_trend(anchors_p):
    
    scalars = {}
    for dow in range(7):
        sub = anchors_p[anchors_p['dow'] == dow].sort_values('day_of_month').reset_index(drop=True)
        if len(sub) < 3:
            scalars[dow] = {int(r['day_of_month']): 1.0 for _, r in sub.iterrows()}
            continue

        x = np.arange(1, len(sub) + 1, dtype=float)
        y = sub['Call Volume'].values.astype(float)
        slope, intercept, r, p_val, se = linregress(x, y)
        r2 = r**2

        if r2 > 0.40 and p_val < 0.10 and abs(slope/y.mean()) > 0.015:
            predicted = intercept + slope * x
            sc = {int(sub.loc[i,'day_of_month']): np.clip(predicted[i]/y.mean(), 0.75, 1.35)
                  for i in range(len(sub))}
        else:
            sc = {int(sub.loc[i,'day_of_month']): 1.0 for i in range(len(sub))}
        scalars[dow] = sc
    return scalars

intramonth = {p: compute_intramonth_trend(anchors[p]) for p in PORTFOLIOS}


#### 9. Identifying Seasonal Ratio

In [10]:
def compute_seasonal_ratio(df_daily_p):
   
    spring24 = df_daily_p[(df_daily_p['year']==2024)&(df_daily_p['month_num'].isin([4,5,6]))].groupby('dow')['Call Volume'].mean()
    aug24    = df_daily_p[(df_daily_p['year']==2024)&(df_daily_p['month_num']==8)].groupby('dow')['Call Volume'].mean()
    spring25 = df_daily_p[(df_daily_p['year']==2025)&(df_daily_p['month_num'].isin([4,5,6]))].groupby('dow')['Call Volume'].mean()
    aug25    = df_daily_p[(df_daily_p['year']==2025)&(df_daily_p['month_num']==8)].groupby('dow')['Call Volume'].mean()
    
    ratio = {}
    for dow in range(7):
        
        r24 = aug24.get(dow, np.nan) / spring24.get(dow, 1.0) if spring24.get(dow,0)>0 else np.nan
        r25 = aug25.get(dow, np.nan) / spring25.get(dow, 1.0) if spring25.get(dow,0)>0 else np.nan
        
        if pd.notna(r24) and pd.notna(r25):
            ratio[dow] = np.clip(0.5*r24 + 0.5*r25, 0.70, 1.40)
        elif pd.notna(r25):
            ratio[dow] = np.clip(r25, 0.70, 1.40)
        elif pd.notna(r24):
            ratio[dow] = np.clip(r24, 0.70, 1.40)
        else:
            ratio[dow] = 1.0
    return ratio

seasonal_ratio = {p: compute_seasonal_ratio(daily[p]) for p in PORTFOLIOS}
for p in PORTFOLIOS:
    print(f"  Portfolio {p}: {[f'{DOW_NAMES[d][:3]}={v:.3f}' for d,v in seasonal_ratio[p].items()]}")

  Portfolio A: ['Mon=0.971', 'Tue=0.926', 'Wed=0.900', 'Thu=0.914', 'Fri=0.955', 'Sat=0.978', 'Sun=1.057']
  Portfolio B: ['Mon=1.044', 'Tue=0.975', 'Wed=0.970', 'Thu=0.958', 'Fri=1.020', 'Sat=1.029', 'Sun=1.109']
  Portfolio C: ['Mon=1.002', 'Tue=0.948', 'Wed=0.957', 'Thu=0.970', 'Fri=0.980', 'Sat=0.983', 'Sun=1.045']
  Portfolio D: ['Mon=1.040', 'Tue=0.963', 'Wed=0.961', 'Thu=0.979', 'Fri=1.002', 'Sat=1.014', 'Sun=1.075']


#### 10. Forecasting Model

In [11]:
def generate_portfolio_forecast(p):
    anc      = anchors[p]
    cv_sh    = cv_shapes[p]
    cct_sh   = cct_shapes[p]
    abd_sh   = abd_shapes[p]
    intra    = intramonth[p]
    uplift   = optimal_uplift[p]
    seas     = seasonal_ratio[p]
    records  = []

    for _, row in anc.iterrows():
        day      = int(row['day_of_month'])
        dow      = int(row['dow'])
        daily_cv = float(row['Call Volume'])
        daily_cct= float(row['CCT']) if pd.notna(row['CCT']) else 320.0
        daily_abd= float(row['Abandon Rate']) if pd.notna(row['Abandon Rate']) else 0.02

        if pd.isna(daily_cv) or daily_cv <= 0:
            continue


        shape_raw = np.array([cv_sh[dow].get(m, 1/48) for m in ALL_SLOTS])
        shape_raw = np.clip(shape_raw, 0, None)
        total = shape_raw.sum()
        shape = shape_raw / total if total > 0 else np.ones(48)/48

        # Slots for CV
        cv_slots = daily_cv * shape

        # Uplifting
        for i, mins in enumerate(ALL_SLOTS):
            hour = mins // 60
            if 8 <= hour < 20:
                cv_slots[i] *= (1 + uplift)
            elif 6 <= hour < 8 or 20 <= hour < 22:
                cv_slots[i] *= (1 + uplift * 0.5)
            

        # CCT Slots
        cct_raw = np.array([cct_sh[dow].get(m, daily_cct) for m in ALL_SLOTS])
        cct_raw = np.clip(cct_raw, 60, 1200)
        shape_mean = cct_raw.mean()
        if shape_mean > 0:
            cct_slots = daily_cct * (cct_raw / shape_mean)
        else:
            cct_slots = np.full(48, daily_cct)
        cct_slots = np.clip(cct_slots, 60, 1200)

        # ABD
        abd_raw = np.array([abd_sh[dow].get(m, 0.0) for m in ALL_SLOTS])
        abd_raw = np.clip(abd_raw, 0, 1)

        # Adding weights 
        cv_weights = shape + 1e-8   
        weighted_mean_abd = np.average(abd_raw, weights=cv_weights)

        if weighted_mean_abd > 0 and daily_abd > 0:
            scale = np.clip(daily_abd / weighted_mean_abd, 0.05, 10.0)
            abd_slots = np.clip(abd_raw * scale, 0, 1)
        elif daily_abd == 0:
            abd_slots = np.zeros(48)
        else:
            abd_slots = abd_raw.copy()

        for i, mins in enumerate(ALL_SLOTS):
            if mins < OVERNIGHT_CUTOFF_AM or mins >= OVERNIGHT_CUTOFF_PM:
                abd_slots[i] = 0.0

        for i, mins in enumerate(ALL_SLOTS):
            cv_v    = max(0.0, float(cv_slots[i]))
            cct_v   = max(0.0, float(cct_slots[i]))
            abd_v   = float(np.clip(abd_slots[i], 0, 1))
            aband   = min(int(round(cv_v * abd_v)), int(np.floor(cv_v)))
            records.append({
                'day_of_month': day, 'dow': dow, 'mins': mins,
                'CV_forecast':  cv_v, 'CCT_forecast': cct_v,
                'ABD_forecast': abd_v, 'Abandoned_forecast': aband,
            })
            

    return pd.DataFrame(records).sort_values(['day_of_month','mins']).reset_index(drop=True)

In [12]:
forecasts = {}
for p in PORTFOLIOS:
    print(f"Portfolio {p}")
    forecasts[p] = generate_portfolio_forecast(p)
    n_days  = forecasts[p]['day_of_month'].nunique()
    n_slots = forecasts[p].groupby('day_of_month').size()
    print(f"{len(forecasts[p])} rows; {n_days} days; slots/day: {n_slots.unique().tolist()}")

for p in PORTFOLIOS:
    df = forecasts[p]
    df['CV_forecast']   = df['CV_forecast'].clip(lower=0)
    df['CCT_forecast']  = df['CCT_forecast'].clip(lower=60, upper=1200)
    df['ABD_forecast']  = df['ABD_forecast'].clip(lower=0, upper=1)
    df['Abandoned_forecast'] = np.minimum(
        df['Abandoned_forecast'],
        df['CV_forecast'].apply(np.floor).astype(int)
    ).clip(lower=0)
    forecasts[p] = df

Portfolio A
1488 rows; 31 days; slots/day: [48]
Portfolio B
1488 rows; 31 days; slots/day: [48]
Portfolio C
1488 rows; 31 days; slots/day: [48]
Portfolio D
1488 rows; 31 days; slots/day: [48]


Creating Output CSV

In [13]:
template = pd.read_csv(TEMPLATE)
template['mins'] = template['Interval'].apply(
    lambda s: int(str(s).split(':')[0])*60 + int(str(s).split(':')[1])
)
template['Day'] = template['Day'].astype(int)

for p in PORTFOLIOS:
    df = forecasts[p][['day_of_month','mins','CV_forecast','CCT_forecast',
                        'ABD_forecast','Abandoned_forecast']].copy()
    df = df.rename(columns={
        'day_of_month':       'Day',
        'CV_forecast':        f'Calls_Offered_{p}',
        'Abandoned_forecast': f'Abandoned_Calls_{p}',
        'ABD_forecast':       f'Abandoned_Rate_{p}',
        'CCT_forecast':       f'CCT_{p}',
    })
    df['Day'] = df['Day'].astype(int)
    df['mins']= df['mins'].astype(int)
    template = template.merge(df, on=['Day','mins'], how='left', suffixes=('_old',''))
    template = template.drop(columns=[c for c in template.columns if c.endswith('_old')])

template = template.sort_values(['Day','mins']).reset_index(drop=True).drop(columns=['mins'])

for p in PORTFOLIOS:
    template[f'Calls_Offered_{p}']   = template[f'Calls_Offered_{p}'].round(2)
    template[f'Abandoned_Calls_{p}'] = template[f'Abandoned_Calls_{p}'].fillna(0).astype(int)
    template[f'Abandoned_Rate_{p}']  = template[f'Abandoned_Rate_{p}'].round(4)
    template[f'CCT_{p}']             = template[f'CCT_{p}'].round(2)


template_col_order = pd.read_csv(TEMPLATE).columns.tolist()
output = template[template_col_order]

In [14]:
#Validation Checks


print("Validation for Submission File")
checks = []
def chk(label, cond, info=''):
    checks.append(cond)
    print(f"[{'correct' if cond else 'check'}] {label}" + (f" ({info})" if info else ""))

chk("Req rows - 1488", len(output)==1488, f"got {len(output)}")
days = sorted(output['Day'].astype(int).unique())
chk("Days in month Aug - 31", days==list(range(1,32)), f"missing: {set(range(1,32))-set(days)}")
chk("Slots/day - 48", (output.groupby('Day')['Interval'].count()==48).all())
chk("no nulls", output.isnull().sum().sum()==0)
chk("No negative CV/CCT", all((output[f'Calls_Offered_{p}']>=0).all() and
    (output[f'CCT_{p}']>=0).all() for p in PORTFOLIOS))
chk("ABD value in [0,1]", all(output[f'Abandoned_Rate_{p}'].between(0,1).all() for p in PORTFOLIOS))


print(f"\n{sum(checks)}/{len(checks)} tests passed")


Validation for Submission File
[correct] Req rows - 1488 (got 1488)
[correct] Days in month Aug - 31 (missing: set())
[correct] Slots/day - 48
[correct] no nulls
[correct] No negative CV/CCT
[correct] ABD value in [0,1]

6/6 tests passed


In [15]:
#Saving output in csv
out_path = OUTPUT_DIR / 'forecast_v39.csv'
output.to_csv(out_path, index=False)
print(f"\ncsv created {out_path} ")



csv created outputs/forecast_v39.csv 
